In [ ]:
import pandas as pd
from time import perf_counter

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join('..')))

In [ ]:
from models import DecisionTree, NaiveBayes
import preprocessing as prp
import visuals as vis

In [ ]:
df = pd.read_csv("../data/raw/mushroom_csv.csv")
df = df.fillna('None')
df.head()

In [ ]:
Y = df['class'].values
X = df.drop(['class'], axis=1).values

X_train, X_test, Y_train, Y_test = prp.train_test_split(X, Y)

In [ ]:
le = prp.LabelEncoder()
oe = prp.OrdinalEncoder()

Y_train_enc = le.fit_transform(Y_train)
Y_test_enc  = le.transform(Y_test)

X_train_enc = oe.fit_transform(X_train)
X_test_enc  = oe.transform(X_test)

In [ ]:
m1 = DecisionTree(max_depth=2, min_gain=0.3) # underfitted
m2 = DecisionTree(max_depth=3, min_gain=0.05) # fitted
m3 = DecisionTree(max_depth=6, min_gain=0.001) # overfitted
m4 = NaiveBayes()

models = [m1, m2, m3, m4]
fit_times = []

for m in models:
    start = perf_counter()
    m.fit(X_train_enc, Y_train_enc)
    end = perf_counter()

    fit_times.append(end - start)


In [ ]:
for i, m in enumerate(models):
    score = m.score(X_test_enc, Y_test_enc)
    print(f"model {i+1}: {score.get('accuracy')}, fitting time: {round(fit_times[i], 4)}, predicting time: {score.get('runtime')}")

In [ ]:
conf_mats = [m.score(X_test_enc, Y_test_enc, scores=["confusion_matrix"], onlyvalues=True)[0] for m in models]

vis.style(style='dark')
vis.plot_confusion_matrices(
    conf_mats,
    ["edible", "poisonous"],
    titling=lambda i: f"Confusion Matrix for Model {i}"
)